### ЗАДАЧА: Панель претензий к поставщикам по недопоставке по паттерну `MVC`

Команда procurement operations разбирает кейсы по недопоставке товара от поставщиков.
После приемки поставки система сравнивает ожидаемое количество и фактически полученный товар.
Если есть расхождение, создается кейс, по которому нужно провести расследование, запросить документы,
согласовать компенсацию от поставщика и закрыть кейс.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за вывод данных;
- `Controller` принимает действия и связывает `Model` и `View`.

## Что должно храниться в кейсе

Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `supplier` — поставщик;
- `shipment_id` — идентификатор поставки;
- `sku` — товар;
- `expected_qty` — ожидаемое количество;
- `received_qty` — фактически принятое количество;
- `unit_cost` — себестоимость единицы товара;
- `claim_qty` — количество товара, которое считается недопоставленным;
- `claim_amount` — сумма претензии к поставщику;
- `approved_compensation` — согласованная компенсация;
- `remaining_loss` — остаток убытка после компенсации;
- `status` — текущий статус кейса;
- `manager` — сотрудник, который ведет кейс;
- `documents_verified` — подтверждены ли документы;
- `decision` — итоговое решение.

## Формулы

При создании кейса и после любого изменения компенсации нужно правильно считать:
- `claim_qty = expected_qty - received_qty`
- если `claim_qty < 0`, нужно выбрасывать ошибку;
- `claim_amount = claim_qty * unit_cost`
- `remaining_loss = claim_amount - approved_compensation`
- все денежные значения нужно округлять до 2 знаков.

## Статусы кейса

- `new`
- `investigating`
- `documents_requested`
- `documents_verified`
- `ready_for_resolution`
- `fully_compensated`
- `partial_compensated`
- `rejected`
- `escalated`

## Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `manager` несуществующему кейсу;
- финальные кейсы (`fully_compensated`, `partial_compensated`, `rejected`, `escalated`) нельзя менять дальше;
- начать расследование можно только из `new` и только если назначен `manager`;
- запросить документы можно только из `investigating`;
- подтвердить документы можно только из `documents_requested`;
- при подтверждении документов поле `documents_verified` должно стать `True`, а статус — `documents_verified`;
- установить `approved_compensation` можно только из `investigating` или `documents_verified`;
- `approved_compensation` не может быть меньше `0`;
- `approved_compensation` не может быть больше `claim_amount`;
- после изменения `approved_compensation` нужно пересчитать `remaining_loss`;
- перевод в `ready_for_resolution` возможен только из `investigating` или `documents_verified`;
- перевод в `ready_for_resolution` невозможен, если `approved_compensation == 0` и при этом `documents_verified == False`;
- завершить кейс как `fully_compensated` можно только из `ready_for_resolution`, если `approved_compensation == claim_amount`;
- завершить кейс как `partial_compensated` можно только из `ready_for_resolution`, если `0 < approved_compensation < claim_amount`;
- завершить кейс как `rejected` можно только из `ready_for_resolution`, если `approved_compensation == 0`;
- эскалировать кейс можно только из `investigating`, `documents_verified` или `ready_for_resolution`;
- при любом финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать менеджера;
- начинать расследование;
- запрашивать документы;
- подтверждать документы;
- устанавливать `approved_compensation`;
- переводить кейс в `ready_for_resolution`;
- завершать кейс как `fully_compensated`;
- завершать кейс как `partial_compensated`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов;
- возвращать summary.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

Если список кейсов пустой, вывести отдельное сообщение.

## Что должен делать `Controller`

Контроллер должен:
- вызывать методы модели;
- оборачивать операции в `try/except` с `ValueError`;
- передавать результат во view;
- обработать все действия из `actions`.

## Формат строки кейса

Каждый кейс можно вывести строкой такого вида:

`case_id | supplier | shipment_id | sku | expected_qty | received_qty | claim_qty | claim_amount | approved_compensation | remaining_loss | status | manager | documents_verified | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_claim_amount` — общая сумма претензий;
- `total_approved_compensation` — общая согласованная компенсация;
- `total_remaining_loss` — общий остаток убытка;
- `verified_docs_cases` — количество кейсов, где документы подтверждены;
- `fully_compensated_amount` — сумма компенсаций по кейсам со статусом `fully_compensated`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить данные из `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести финальное состояние кейсов и summary.

In [ ]:
initial_cases = [
    ("SC-100", "alpha-supply", "SHIP-7001", "SKU-100", 120, 102, 35.0),
    ("SC-101", "beta-distribution", "SHIP-7002", "SKU-200", 80, 80, 50.0),
]

actions = [
    ("show",),
    ("investigate", "SC-100"),
    ("assign", "SC-100", "Olga"),
    ("investigate", "SC-100"),
    ("docs_request", "SC-100"),
    ("docs_verify", "SC-100"),
    ("set_compensation", "SC-100", 500.0),
    ("ready", "SC-100"),
    ("partial", "SC-100", "supplier_accepted_partial_claim"),
    ("create", "SC-102", "gamma-warehouses", "SHIP-7003", "SKU-300", 60, 40, 28.0),
    ("assign", "SC-102", "Max"),
    ("investigate", "SC-102"),
    ("set_compensation", "SC-102", 560.0),
    ("ready", "SC-102"),
    ("full", "SC-102", "full_compensation_approved"),
    ("create", "SC-103", "delta-trade", "SHIP-7004", "SKU-400", 45, 30, 42.0),
    ("assign", "SC-103", "Ina"),
    ("investigate", "SC-103"),
    ("escalate", "SC-103", "supplier_disputes_shortage"),
    ("show",),
]

class ClaimCase:
    def __init__(self, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost):
        if expected_qty < received_qty:
            raise ValueError("Expected quantity cannot be less than received quantity")
        self.case_id = case_id
        self.supplier = supplier
        self.shipment_id = shipment_id
        self.sku = sku
        self.expected_qty = expected_qty
        self.received_qty = received_qty
        self.unit_cost = unit_cost
        self.claim_qty = expected_qty - received_qty
        self.claim_amount = round(self.claim_qty * unit_cost, 2)
        self.approved_compensation = 0.0
        self.remaining_loss = self.claim_amount
        self.status = "new"
        self.manager = None
        self.documents_verified = False
        self.decision = None

    def update_compensation(self, amount):
        if amount < 0:
            raise ValueError("Compensation cannot be negative")
        if amount > self.claim_amount:
            raise ValueError("Compensation cannot exceed claim amount")
        self.approved_compensation = round(amount, 2)
        self.remaining_loss = round(self.claim_amount - self.approved_compensation, 2)

    def to_string(self):
        return (f"{self.case_id} | {self.supplier} | {self.shipment_id} | "
                f"{self.sku} | {self.expected_qty} | {self.received_qty} | "
                f"{self.claim_qty} | {self.claim_amount} | "
                f"{self.approved_compensation} | {self.remaining_loss} | "
                f"{self.status} | {self.manager or ''} | {self.documents_verified} | "
                f"{self.decision or ''}")


class ClaimsModel:
    FINAL_STATUSES = {"fully_compensated", "partial_compensated", "rejected", "escalated"}


    def __init__(self):
        self.cases = {}

    def create_case(self, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost):
        if case_id in self.cases:
            raise ValueError(f"Case with ID {case_id} already exists")
        case = ClaimCase(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
        self.cases[case_id] = case

    def assign_manager(self, case_id, manager):
        case = self._get_case(case_id)
        if case.status in self.FINAL_STATUSES:
            raise ValueError(f"Cannot modify final case {case_id}")
        case.manager = manager

    def investigate(self, case_id):
        case = self._get_case(case_id)
        if case.status in self.FINAL_STATUSES:
            raise ValueError(f"Cannot modify final case {case_id}")
        if case.status != "new":
            raise ValueError(f"Can only start investigation from 'new' status")
        if not case.manager:
            raise ValueError(f"Manager must be assigned before investigation")
        case.status = "investigating"

    def request_documents(self, case_id):
        case = self._get_case(case_id)
        if case.status in self.FINAL_STATUSES:
            raise ValueError(f"Cannot modify final case {case_id}")
        if case.status != "investigating":
            raise ValueError(f"Can only request documents from 'investigating' status")
        case.status = "documents_requested"

    def verify_documents(self, case_id):
        case = self._get_case(case_id)
        if case.status in self.FINAL_STATUSES:
            raise ValueError(f"Cannot modify final case {case_id}")
        if case.status != "documents_requested":
            raise ValueError(f"Can only verify documents from 'documents_requested' status")
        case.documents_verified = True
        case.status = "documents_verified"

    def set_compensation(self, case_id, amount):
        case = self._get_case(case_id)
        if case.status in self.FINAL_STATUSES:
            raise ValueError(f"Cannot modify final case {case_id}")
        if case.status not in ["investigating", "documents_verified"]:
            raise ValueError(f"Can only set compensation from 'investigating' or 'documents_verified' status")
        case.update_compensation(amount)

    def ready_for_resolution(self, case_id):
        case = self._get_case(case_id)
        if case.status in self.FINAL_STATUSES:
            raise ValueError(f"Cannot modify final case {case_id}")
        if case.status not in ["investigating", "documents_verified"]:
            raise ValueError(f"Can only move to ready_for_resolution from 'investigating' or 'documents_verified'")
        if case.approved_compensation == 0 and not case.documents_verified:
            raise ValueError(f"Cannot move to ready_for_resolution: no compensation and documents not verified")
        case.status = "ready_for_resolution"

    def fully_compensated(self, case_id, decision):
        case = self._get_case(case_id)
        if case.status != "ready_for_resolution":
            raise ValueError(f"Can only fully compensate from 'ready_for_resolution' status")
        if case.approved_compensation != case.claim_amount:
            raise ValueError(f"Compensation must equal claim amount for full compensation")
        case.status = "fully_compensated"
        case.decision = decision


    def partial_compensated(self, case_id, decision):
        case = self._get_case(case_id)
        if case.status != "ready_for_resolution":
            raise ValueError(f"Can only partially compensate from 'ready_for_resolution' status")
        if not (0 < case.approved_compensation < case.claim_amount):
            raise ValueError(f"Invalid compensation amount for partial compensation")
        case.status = "partial_compensated"
        case.decision = decision

    def rejected(self, case_id, decision):
        case = self._get_case(case_id)
        if case.status != "ready_for_resolution":
            raise ValueError(f"Can only reject from 'ready_for_resolution' status")
        if case.approved_compensation != 0:
            raise ValueError(f"Compensation must be 0 for rejected case")
        case.status = "rejected"
        case.decision = decision

    def escalate(self, case_id, decision):
        case = self._get_case(case_id)
        if case.status in self.FINAL_STATUSES:
            raise ValueError(f"Cannot modify final case {case_id}")
        if case.status not in ["investigating", "documents_verified", "ready_for_resolution"]:
            raise ValueError(f"Can only escalate from specific statuses")
        case.status = "escalated"
        case.decision = decision

    
    def get_cases(self):
        return list(self.cases.values())

    def get_summary(self):
        summary = {
            "status_counts": {},
            "total_claim_amount": 0.0,
            "total_approved_compensation": 0.0,
            "total_remaining_loss": 0.0,
            "verified_docs_cases": 0,
            "fully_compensated_amount": 0.0
        }
        for case in self.cases.values():
            status = case.status
            summary["status_counts"][status] = summary["status_counts"].get(status, 0) + 1
            summary["total_claim_amount"] += case.claim_amount
            summary["total_approved_compensation"] += case.approved_compensation
            summary["total_remaining_loss"] += case.remaining_loss
            if case.documents_verified:
                summary["verified_docs_cases"] += 1
            if case.status == "fully_compensated":
                summary["fully_compensated_amount"] += case.approved_compensation

        # Округление всех итоговых значений после цикла
        summary["total_claim_amount"] = round(summary["total_claim_amount"], 2)
        summary["total_approved_compensation"] = round(summary["total_approved_compensation"], 2)
        summary["total_remaining_loss"] = round(summary["total_remaining_loss"], 2)
        summary["fully_compensated_amount"] = round(summary["fully_compensated_amount"], 2)

        return summary

    def _get_case(self, case_id):
        if case_id not in self.cases:
            raise ValueError(f"Case with ID {case_id} does not exist")
        return self.cases[case_id]


class ClaimsView:
    @staticmethod
    def show_cases(cases):
        if not cases:
            print("No cases available.")
            return
        print("Case list:")
        for case in cases:
            print(case.to_string())
        print()

    @staticmethod
    def show_summary(summary):
        print("Summary:")
        print(f"Status counts: {summary['status_counts']}")
        print(f"Total claim amount: {summary['total_claim_amount']}")
        print(f"Total approved compensation: {summary['total_approved_compensation']}")
        print(f"Total remaining loss: {summary['total_remaining_loss']}")
        print(f"Verified documents cases: {summary['verified_docs_cases']}")
        print(f"Fully compensated amount: {summary['fully_compensated_amount']}")
        print()

    @staticmethod
    def show_success(message):
        print(f"SUCCESS: {message}")

    @staticmethod
    def show_error(error_message):
        print(f"ERROR: {error_message}")

class ClaimsController:
    def __init__(self, model, view):
        self.model = model
        self.view = view

    def handle_action(self, action):
        try:
            action_type = action[0]
            if action_type == "show":
                cases = self.model.get_cases()
                self.view.show_cases(cases)
            elif action_type == "create":
                _, case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost = action
                self.model.create_case(case_id, supplier, shipment_id, sku, expected_qty, received_qty, unit_cost)
                self.view.show_success(f"Case {case_id} created successfully")
            elif action_type == "assign":
                _, case_id, manager = action
                self.model.assign_manager(case_id, manager)
                self.view.show_success(f"Manager {manager} assigned to case {case_id}")
            elif action_type == "investigate":
                _, case_id = action
                self.model.investigate(case_id)
                self.view.show_success(f"Investigation started for case {case_id}")
            elif action_type == "docs_request":
                _, case_id = action
                self.model.request_documents(case_id)
                self.view.show_success(f"Documents requested for case {case_id}")
            elif action_type == "docs_verify":
                _, case_id = action
                self.model.verify_documents(case_id)
                self.view.show_success(f"Documents verified for case {case_id}")
            elif action_type == "set_compensation":
                _, case_id, amount = action
                self.model.set_compensation(case_id, amount)
                self.view.show_success(f"Compensation {amount} set for case {case_id}")
            elif action_type == "ready":
                _, case_id = action
                self.model.ready_for_resolution(case_id)
                self.view.show_success(f"Case {case_id} moved to ready_for_resolution")
            elif action_type == "full":
                _, case_id, decision = action
                self.model.fully_compensated(case_id, decision)
                self.view.show_success(f"Case {case_id} fully compensated")
            elif action_type == "partial":
                _, case_id, decision = action
                self.model.partial_compensated(case_id, decision)
                self.view.show_success(f"Case {case_id} partially compensated")
            elif action_type == "reject":
                _, case_id, decision = action
                self.model.rejected(case_id, decision)
                self.view.show_success(f"Case {case_id} rejected")
            elif action_type == "escalate":
                _, case_id, decision = action
                self.model.escalate(case_id, decision)
                self.view.show_success(f"Case {case_id} escalated")
            else:
                self.view.show_error(f"Unknown action: {action_type}")
        except ValueError as e:
            self.view.show_error(str(e))

    def process_actions(self, actions):
        for action in actions:
            self.handle_action(action)


# Инициализация компонентов MVC
model = ClaimsModel()
view = ClaimsView()
controller = ClaimsController(model, view)


# Загрузка начальных кейсов
for case_data in initial_cases:
    try:
        model.create_case(*case_data)
        view.show_success(f"Initial case {case_data[0]} loaded")
    except ValueError as e:
        view.show_error(str(e))

# Обработка действий
controller.process_actions(actions)

# Финальный вывод
final_cases = model.get_cases()
view.show_cases(final_cases)
summary = model.get_summary()
view.show_summary(summary)
